# EEG Emotion Recognition - Notebook 05: Proposed Hybrid CNN-BiLSTM-Attention Model

## Proposed Main Architecture
```
Input EEG Signal: (batch_size, 256, 32)
       |
       v
32 EEG Channels (Preprocessed & Zero-Leakage Normalized)
       |
       v
Phase 1: CNN Feature Extraction
  - Conv1D(32, k=5, pad='same') + BatchNorm + ReLU + MaxPool1D(2) + Dropout(0.2)  -> (B, 128, 32)
  - Conv1D(64, k=3, pad='same') + BatchNorm + ReLU + MaxPool1D(2) + Dropout(0.2)  -> (B, 64, 64)
       |
       v
Phase 2: BiLSTM Temporal Dependency Learning
  - Bidirectional(LSTM(64, return_seq=True, dropout=0.25)) -> (B, 64, 128)
  - Bidirectional(LSTM(32, return_seq=True, dropout=0.25)) -> (B, 64, 64)
       |
       v
Phase 3: Temporal Bahdanau-Style Self-Attention Mechanism
  - u_t = tanh(W * h_t + b)
  - alpha_t = softmax(u_t^T * v)
  - c = sum(alpha_t * h_t) -> (B, 64)
       |
       v
Phase 4: Fully Connected Regression Head
  - Dense(64, relu) + Dropout(0.3)
  - Dense(32, relu) + Dropout(0.2)
  - Dense(4, linear) -> [Valence, Arousal, Dominance, Liking]
```

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/DEAP/processed'
    CHECKPOINT_DIR = '/content/drive/MyDrive/DEAP/checkpoints/hybrid'
else:
    DATA_DIR = '../data/processed' if os.path.exists('../data/processed') else './data/processed'
    CHECKPOINT_DIR = '../models/checkpoints/hybrid' if os.path.exists('../models') else './models/checkpoints/hybrid'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CONFIG
print("Environment initialized successfully.")

In [ ]:
# Load Preprocessed Data Splits
train_data = np.load(os.path.join(DATA_DIR, 'train_data.npz'))
val_data = np.load(os.path.join(DATA_DIR, 'val_data.npz'))
test_data = np.load(os.path.join(DATA_DIR, 'test_data.npz'))

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val = val_data['X'], val_data['y']
X_test, y_test = test_data['X'], test_data['y']

print(f"Training set:   X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Testing set:    X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Build and Compile Proposed Hybrid Architecture
try:
    import tensorflow as tf
    from src.models.hybrid_model import build_keras_hybrid
    
    # Proposed Model with Attention
    model_hybrid = build_keras_hybrid(input_shape=(256, 32), num_outputs=4, use_attention=True, dropout_rate=0.3)
    model_hybrid.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG.training.learning_rate),
        loss=tf.keras.losses.Huber(delta=CONFIG.training.huber_delta),
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    model_hybrid.summary()
except ImportError:
    print("TensorFlow not installed locally. Notebook is ready for execution on Google Colab T4 GPU.")

In [ ]:
# Train Proposed Hybrid Model with Early Stopping & LR Decay
if 'model_hybrid' in locals():
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=CONFIG.training.early_stopping_patience,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=CONFIG.training.reduce_lr_factor,
            patience=CONFIG.training.reduce_lr_patience,
            min_lr=CONFIG.training.min_lr
        ),
        tf.keras.callbacks.ModelCheckpoint(
            os.path.join(CHECKPOINT_DIR, 'best_hybrid_attention_model.keras'),
            save_best_only=True
        )
    ]
    
    history = model_hybrid.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=CONFIG.training.batch_size,
        epochs=CONFIG.training.epochs,
        callbacks=callbacks,
        verbose=1
    )